## BASELINE MODELS

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    StratifiedGroupKFold,
    cross_validate
)

from sklearn.svm import SVC

from sklearn.ensemble import RandomForestClassifier

from sklearn.dummy import DummyClassifier


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

processed_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "primary_532nm_features.npz"
)

data = np.load(processed_path)

X_max = data["X_max"]
X_l2 = data["X_l2"]
X_area = data["X_area"]

grid = data["raman_grid"]

mineral = data["mineral"]
rruff_id = data["rruff_id"]

feature_metadata = pd.DataFrame({
    "mineral": mineral,
    "rruff_id": rruff_id
})

print("Original feature matrix:", X_l2.shape)
print("Number of minerals:", feature_metadata["mineral"].nunique())
print("Unique sample IDs:", feature_metadata["rruff_id"].nunique())

In [ ]:
split_path = (
    PROJECT_ROOT
    / "data"
    / "manifests"
    / "primary_532nm_split_assignment.csv"
)

split_assignment = pd.read_csv(split_path)

print(split_assignment["split"].value_counts())

In [ ]:
train_ids = set(
    split_assignment.loc[
        split_assignment["split"] == "train",
        "rruff_id"
    ]
)

train_mask = feature_metadata["rruff_id"].isin(train_ids).to_numpy()

X_train = {
    "Maximum": X_max[train_mask],
    "L2": X_l2[train_mask],
    "Area": X_area[train_mask]
}

meta_train = (
    feature_metadata.loc[train_mask]
    .reset_index(drop=True)
)

y_train = meta_train["mineral"].to_numpy()

groups_train = meta_train["rruff_id"].to_numpy()

print("Training observations:", len(y_train))
print("Training classes:", len(np.unique(y_train)))
print("Training sample IDs:", len(np.unique(groups_train)))


In [ ]:
assert all(
    matrix.shape == (82, 1991)
    for matrix in X_train.values()
)

assert len(y_train) == 82

assert len(np.unique(y_train)) == 12

assert len(np.unique(groups_train)) == 82

assert all(
    np.isfinite(matrix).all()
    for matrix in X_train.values()
)

assert set(groups_train) == train_ids

## 1. Create the classifiers

#### LINEAR SVM

In [ ]:
linear_svm = SVC(
    kernel="linear",
    C=1.0,
    class_weight="balanced"
)

#### RBF SVM

In [ ]:
rbf_svm = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    class_weight="balanced"
)

#### RANDOM FORESTS

In [ ]:
random_forest = RandomForestClassifier(
    n_estimators=500,
    max_features="sqrt",
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

#### DUMMY CLASSIFIER

In [ ]:
dummy = DummyClassifier(
    strategy="most_frequent"
)

In [ ]:
models = {
    "Dummy": dummy,
    "Linear SVM": linear_svm,
    "RBF SVM": rbf_svm,
    "Random Forest": random_forest
}

### 1.2 Five fold cross-validation

In [ ]:
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
scoring = {
    "macro_f1": "f1_macro",
    "balanced_accuracy": "balanced_accuracy",
    "accuracy": "accuracy"
}

In [ ]:
results = []

for normalization_name, X in X_train.items():

    print(f"\nNormalisation: {normalization_name}")

    for model_name, model in models.items():

        print(f"  Evaluating: {model_name}")

        cv_result = cross_validate(
            estimator=model,
            X=X,
            y=y_train,
            groups=groups_train,
            cv=cv,
            scoring=scoring,
            return_train_score=True,
            n_jobs=1,
            error_score="raise"
        )

        results.append({

            "normalization": normalization_name,

            "model": model_name,

            "train_macro_f1_mean":
                cv_result["train_macro_f1"].mean(),

            "validation_macro_f1_mean":
                cv_result["test_macro_f1"].mean(),

            "validation_macro_f1_std":
                cv_result["test_macro_f1"].std(ddof=1),

            "validation_balanced_accuracy_mean":
                cv_result["test_balanced_accuracy"].mean(),

            "validation_balanced_accuracy_std":
                cv_result["test_balanced_accuracy"].std(ddof=1),

            "validation_accuracy_mean":
                cv_result["test_accuracy"].mean(),

            "validation_accuracy_std":
                cv_result["test_accuracy"].std(ddof=1)
        })

In [ ]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "validation_macro_f1_mean",
    ascending=False
).reset_index(drop=True)

display(
    results_df.round(3)
)

In [ ]:
reports_dir = PROJECT_ROOT / "reports"

reports_dir.mkdir(exist_ok=True)

results_df.to_csv(
    reports_dir / "baseline_cv_results.csv",
    index=False
)

### 1.3 Visualise the baseline comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for normalization_name in ["Maximum", "L2", "Area"]:

    subset = results_df[
        results_df["normalization"] == normalization_name
    ]

    subset = subset.set_index("model")

    ax.plot(
        subset.index,
        subset["validation_macro_f1_mean"],
        marker="o",
        linewidth=1.5,
        label=normalization_name
    )

ax.set_title(
    "Baseline model comparison — 5-fold cross-validation"
)

ax.set_ylabel("Mean validation Macro-F1")

ax.set_xlabel("Classifier")

ax.set_ylim(0, 1)

ax.grid(alpha=0.25)

ax.legend(title="Normalisation")

plt.xticks(rotation=15)

plt.tight_layout()

fig.savefig(
    reports_dir / "baseline_model_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 2. Cross-validation error analysis

We analyse out-of-fold predictions for the baseline models.

The objective is to investigate:
- Fold-to-fold performance variability.
- Per-class precision and recall.
- Confusion between related mineral species.
- Prediction agreement across normalisation methods.

In [ ]:
from sklearn.base import clone

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.model_selection import StratifiedGroupKFold

In [ ]:
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_folds = list(
    cv.split(
        X_train["L2"],
        y_train,
        groups=groups_train
    )
)

print("Number of folds:", len(cv_folds))

for fold, (train_idx, val_idx) in enumerate(cv_folds, start=1):

    print(f"\nFold {fold}")

    print("Training observations:", len(train_idx))
    print("Validation observations:", len(val_idx))

    train_groups = set(groups_train[train_idx])
    validation_groups = set(groups_train[val_idx])

    assert train_groups.isdisjoint(validation_groups)

In [ ]:
diagnostic_results = []

oof_predictions = {}

oof_folds = {}

for normalization_name, X in X_train.items():

    for model_name in [
        "Linear SVM",
        "RBF SVM",
        "Random Forest"
    ]:

        config_name = f"{model_name} | {normalization_name}"

        estimator = models[model_name]

        print("Evaluating:", config_name)

        predictions = np.empty(
            len(y_train),
            dtype=object
        )

        fold_assignment = np.full(
            len(y_train),
            -1,
            dtype=int
        )

        seen = np.zeros(
            len(y_train),
            dtype=bool
        )

        for fold, (train_idx, val_idx) in enumerate(
            cv_folds,
            start=1
        ):

            model = clone(estimator)

            model.fit(
                X[train_idx],
                y_train[train_idx]
            )

            y_pred = model.predict(
                X[val_idx]
            )

            predictions[val_idx] = y_pred

            fold_assignment[val_idx] = fold

            seen[val_idx] = True

            score = f1_score(
                y_train[val_idx],
                y_pred,
                labels=np.unique(y_train),
                average="macro",
                zero_division=0
            )

            diagnostic_results.append({

                "configuration": config_name,

                "fold": fold,

                "macro_f1": score,

                "accuracy": accuracy_score(
                    y_train[val_idx],
                    y_pred
                ),

                "balanced_accuracy": balanced_accuracy_score(
                    y_train[val_idx],
                    y_pred
                )
            })

        assert seen.all()

        assert np.all(fold_assignment > 0)

        oof_predictions[config_name] = predictions

        oof_folds[config_name] = fold_assignment


fold_results = pd.DataFrame(diagnostic_results)

display(
    fold_results.pivot(
        index="fold",
        columns="configuration",
        values="macro_f1"
    ).round(3)
)

In [ ]:
rf_max = oof_predictions["Random Forest | Maximum"]

rf_l2 = oof_predictions["Random Forest | L2"]

rf_area = oof_predictions["Random Forest | Area"]

print(
    "Maximum versus L2 agreement:",
    np.mean(rf_max == rf_l2)
)

print(
    "Maximum versus Area agreement:",
    np.mean(rf_max == rf_area)
)

print(
    "L2 versus Area agreement:",
    np.mean(rf_l2 == rf_area)
)

In [ ]:
selected_configs = [
    "Random Forest | L2",
    "Linear SVM | Maximum",
    "RBF SVM | L2"
]

class_reports = {}

for config in selected_configs:

    predictions = oof_predictions[config]

    report = classification_report(
        y_train,
        predictions,
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(report).T

    class_reports[config] = report_df

    print("\n", "=" * 70)
    print(config)

    display(
        report_df.round(3)
    )

In [ ]:
reports_dir = PROJECT_ROOT / "reports"
figures_dir = reports_dir / "figures"

figures_dir.mkdir(
    parents=True,
    exist_ok=True
)

mineral_order = [
    "Grossular",
    "Andradite",
    "Diopside",
    "Epidote",
    "Forsterite",
    "Enstatite",
    "Elbaite",
    "Beryl",
    "Fluorapatite",
    "Almandine",
    "Pyrope",
    "Wendwilsonite"
]

for config in selected_configs:

    predictions = oof_predictions[config]

    fig, ax = plt.subplots(figsize=(11, 9))

    ConfusionMatrixDisplay.from_predictions(
        y_train,
        predictions,
        labels=mineral_order,
        display_labels=mineral_order,
        normalize="true",
        xticks_rotation=45,
        cmap="Blues",
        values_format=".2f",
        ax=ax
    )

    ax.set_title(
        f"Cross-validation confusion matrix\n{config}"
    )

    plt.tight_layout()

    filename = (
        config.lower()
        .replace(" ", "_")
        .replace("|", " ")
        .strip()
        .replace("__", "_")
    )

    fig.savefig(
        figures_dir / f"confusion_{filename}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
best_config = "Random Forest | L2"

predictions = oof_predictions[best_config]

errors = pd.DataFrame({

    "rruff_id": meta_train["rruff_id"],

    "mineral_true": y_train,

    "mineral_predicted": predictions,

    "fold": oof_folds[best_config]

})

errors = errors[
    errors["mineral_true"] != errors["mineral_predicted"]
].copy()

print("Total misclassified observations:", len(errors))

display(
    errors.sort_values(
        ["mineral_true", "mineral_predicted"]
    )
)

In [ ]:
confusion_pairs = (
    errors
    .groupby(
        ["mineral_true", "mineral_predicted"]
    )
    .size()
    .sort_values(ascending=False)
)

display(
    confusion_pairs.to_frame(
        "n_errors"
    )
)

#### 2.1 Verify that our normalisation matrices are genuinely different
I also want one quick sanity check because all three Random Forest models made identical predictions.

In [ ]:
normalization_matrices = {
    "Maximum": X_train["Maximum"],
    "L2": X_train["L2"],
    "Area": X_train["Area"]
}

for name, matrix in normalization_matrices.items():

    print(f"\n{name}")

    print("Shape:", matrix.shape)

    print(
        "Average row L2 norm:",
        np.linalg.norm(matrix, axis=1).mean()
    )

    print(
        "Average maximum intensity:",
        matrix.max(axis=1).mean()
    )

In [ ]:
for first, second in [
    ("Maximum", "L2"),
    ("Maximum", "Area"),
    ("L2", "Area")
]:

    difference = (
        normalization_matrices[first]
        - normalization_matrices[second]
    )

    print(
        f"{first} versus {second}:",
        "maximum absolute difference =",
        np.max(np.abs(difference))
    )

## 3. Hyperparameter tuning

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

In [ ]:

from sklearn.model_selection import GridSearchCV

search_spaces = {

    "Linear SVM": {

        "X": X_train["Maximum"],

        "estimator": SVC(
            kernel="linear",
            class_weight="balanced"
        ),

        "parameters": {
            "C": [0.01, 0.1, 1, 10, 100, 1000]
        }
    },

    "RBF SVM": {

        "X": X_train["L2"],

        "estimator": SVC(
            kernel="rbf",
            class_weight="balanced"
        ),

        "parameters": {
            "C": [0.1, 1, 10, 100, 1000],
            "gamma": ["scale", 0.01, 0.1, 1, 10]
        }
    },

    "Random Forest": {

        "X": X_train["L2"],

        "estimator": RandomForestClassifier(
            n_estimators=400,
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=1
        ),

        "parameters": {
            "max_depth": [None, 5, 10],
            "min_samples_leaf": [1, 2, 3],
            "max_features": ["sqrt", 0.1, 0.3]
        }
    }
}


In [ ]:

best_models = {}
search_objects = {}
search_summaries = []

for model_name, specification in search_spaces.items():

    print(f"\nOptimising {model_name}...")

    search = GridSearchCV(
        estimator=specification["estimator"],

        param_grid=specification["parameters"],

        scoring="f1_macro",

        cv=cv_folds,

        refit=True,

        return_train_score=True,

        n_jobs=-1,

        error_score="raise"
    )

    search.fit(
        specification["X"],
        y_train
    )

    best_models[model_name] = search.best_estimator_

    search_objects[model_name] = search

    best_index = search.best_index_

    search_summaries.append({

        "model": model_name,

        "best_cv_macro_f1": search.best_score_,

        "cv_std": search.cv_results_[
            "std_test_score"
        ][best_index],

        "train_macro_f1": search.cv_results_[
            "mean_train_score"
        ][best_index],

        "best_parameters": search.best_params_

    })

    print("Best Macro-F1:", search.best_score_)

    print("Best parameters:", search.best_params_)


In [ ]:
tuning_summary = pd.DataFrame(search_summaries)

display(
    tuning_summary.sort_values(
        "best_cv_macro_f1",
        ascending=False
    )
)

In [ ]:
for model_name, search in search_objects.items():

    print("\n" + "=" * 60)
    print(model_name)

    top_results = (
        pd.DataFrame(search.cv_results_)
        .sort_values("rank_test_score")
        [
            [
                "mean_train_score",
                "mean_test_score",
                "std_test_score",
                "params"
            ]
        ]
        .head(5)
    )

    display(top_results.round(3))

In [ ]:
tuning_summary.to_csv(
    PROJECT_ROOT / "reports" /
    "hyperparameter_tuning_summary.csv",
    index=False
)

for model_name, search in search_objects.items():

    results = pd.DataFrame(search.cv_results_)

    filename = (
        model_name.lower().replace(" ", "_")
        + "_grid_search.csv"
    )

    results.to_csv(
        PROJECT_ROOT / "reports" / filename,
        index=False
    )

In [ ]:
final_candidate = SVC(
    kernel="rbf",
    C=1000,
    gamma=0.1,
    class_weight="balanced",
    random_state=42
)

### 4. Keras 1D CNN

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

import keras
from keras import layers, regularizers


from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import ConfusionMatrixDisplay

In [ ]:
X_cnn = X_train["L2"]

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y_train)

class_names = label_encoder.classes_

X_cnn = X_cnn[..., np.newaxis]

print("CNN input:", X_cnn.shape)
print("Number of classes:", len(class_names))

In [ ]:
def build_cnn(input_length, n_classes):

    inputs = keras.Input(
        shape=(input_length, 1)
    )

    x = layers.Conv1D(
        filters=16,
        kernel_size=9,
        padding="same",
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4)
    )(inputs)

    x = layers.MaxPooling1D(
        pool_size=4
    )(x)

    x = layers.Conv1D(
        filters=32,
        kernel_size=7,
        padding="same",
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)

    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dropout(0.4)(x)

    outputs = layers.Dense(
        n_classes,
        activation="softmax"
    )(x)

    model = keras.Model(
        inputs=inputs,
        outputs=outputs
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=1e-3
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
from sklearn.metrics import balanced_accuracy_score

cnn_oof_predictions = np.empty(
    len(y_encoded),
    dtype=int
)

cnn_fold_results = []

for fold, (outer_train_idx, outer_val_idx) in enumerate(
    cv_folds,
    start=1
):

    print(f"Training fold {fold}/5")

    tf.keras.backend.clear_session()

    tf.keras.utils.set_random_seed(100 + fold)

    # Split the outer training observations for early stopping
    inner_train_idx, inner_val_idx = train_test_split(
        outer_train_idx,
        test_size=0.20,
        random_state=100 + fold,
        stratify=y_encoded[outer_train_idx]
    )

    # Explicit group-leakage checks
    assert set(groups_train[inner_train_idx]).isdisjoint(
        set(groups_train[inner_val_idx])
    )

    assert set(groups_train[outer_train_idx]).isdisjoint(
        set(groups_train[outer_val_idx])
    )

    model = build_cnn(
        input_length=X_cnn.shape[1],
        n_classes=len(class_names)
    )

    callbacks = [

        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=12,
            restore_best_weights=True
        ),

        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-5
        )
    ]

    history = model.fit(
        X_cnn[inner_train_idx],
        y_encoded[inner_train_idx],

        validation_data=(
            X_cnn[inner_val_idx],
            y_encoded[inner_val_idx]
        ),

        epochs=120,
        batch_size=8,
        callbacks=callbacks,
        verbose=0
    )

    probabilities = model.predict(
        X_cnn[outer_val_idx],
        verbose=0
    )

    predictions = probabilities.argmax(axis=1)

    cnn_oof_predictions[outer_val_idx] = predictions

    fold_f1 = f1_score(
        y_encoded[outer_val_idx],
        predictions,
        average="macro",
        zero_division=0
    )

    fold_balanced_accuracy = balanced_accuracy_score(
        y_encoded[outer_val_idx],
        predictions
    )

    best_epoch = (
        np.argmin(history.history["val_loss"]) + 1
    )

    cnn_fold_results.append({

        "fold": fold,

        "macro_f1": fold_f1,

        "balanced_accuracy": fold_balanced_accuracy,

        "best_epoch": best_epoch

    })

cnn_results = pd.DataFrame(cnn_fold_results)

display(cnn_results.round(3))

print(
    "Mean CNN Macro-F1:",
    cnn_results["macro_f1"].mean()
)

print(
    "Std CNN Macro-F1:",
    cnn_results["macro_f1"].std(ddof=1)
)

In [ ]:
cnn_oof_labels = label_encoder.inverse_transform(
    cnn_oof_predictions
)

cnn_report = classification_report(
    y_train,
    cnn_oof_labels,
    output_dict=True,
    zero_division=0
)

display(
    pd.DataFrame(cnn_report).T.round(3)
)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))

ConfusionMatrixDisplay.from_predictions(
    y_train,
    cnn_oof_labels,
    labels=class_names,
    display_labels=class_names,
    normalize="true",
    xticks_rotation=45,
    cmap="Blues",
    values_format=".2f",
    ax=ax
)

ax.set_title("Keras 1D CNN — Cross-validation confusion matrix")

plt.tight_layout()

fig.savefig(
    PROJECT_ROOT / "reports" / "figures" /
    "cnn_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
cnn_results.to_csv(
    PROJECT_ROOT / "reports" /
    "cnn_cv_results.csv",
    index=False
)

In [ ]:
final_model = SVC(
    kernel="rbf",
    C=1000,
    gamma=0.1,
    class_weight="balanced",
    probability=True,
    random_state=42
)